# 01 - Image Basics and Preprocessing

This notebook introduces the Phase 1 synthetic FSOC images and Phase 2 preprocessing output. The goal is to understand what the virtual camera frames look like before candidate detection.

Pipeline covered here:

`Original BGR image -> grayscale -> optional contrast normalization -> Gaussian blur -> thresholding -> morphology -> binary image`

In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import PreprocessingConfig, load_image, preprocess_frame

IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "python-generated"
LABELS_PATH = PROJECT_ROOT / "data" / "labels" / "labels.csv"

labels = pd.read_csv(LABELS_PATH)
labels.head()

## Scenario Distribution

The generator creates six scenario types so later stages can handle clean frames, stars, sensor noise, false beacons, and target-absent cases.

In [ ]:
labels.groupby(["scenario_id", "scenario_type"]).size().rename("count")

## Show One Example From Each Scenario

In [ ]:
samples = labels.groupby("scenario_id", sort=True).head(1).reset_index(drop=True)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, row in zip(axes.flatten(), samples.itertuples(index=False)):
    image = load_image(IMAGE_DIR / row.filename)
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    ax.imshow(rgb)
    if row.target_visible:
        ax.scatter([row.target_x], [row.target_y], c="lime", marker="x", s=80)
        ax.set_title(f"S{row.scenario_id}: {row.scenario_type}\nTarget ({row.target_x}, {row.target_y})")
    else:
        ax.set_title(f"S{row.scenario_id}: {row.scenario_type}\nTarget absent")
    ax.axis("off")
plt.tight_layout()

## Inspect Pixel Values

Most pixels are dark background. The beacon and false bright objects appear as small high-intensity regions.

In [ ]:
row = labels.iloc[0]
image = load_image(IMAGE_DIR / row.filename)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

print("shape:", image.shape)
print("dtype:", image.dtype)
print("min / mean / max intensity:", int(gray.min()), float(gray.mean()), int(gray.max()))

plt.figure(figsize=(7, 4))
plt.hist(gray.ravel(), bins=50)
plt.title("Grayscale Intensity Histogram")
plt.xlabel("Intensity")
plt.ylabel("Pixel count")
plt.grid(alpha=0.25)

## Compare Threshold Methods

For the current synthetic dataset, fixed thresholding around `180` gave perfect candidate recall with much less clutter than Otsu.

In [ ]:
example_name = "synthetic_0300_sensor_noise_and_blur.png"
frame = load_image(IMAGE_DIR / example_name)

configs = [
    PreprocessingConfig(threshold_method="fixed", threshold_value=180, blur_kernel=5, morph_kernel=3),
    PreprocessingConfig(threshold_method="fixed", threshold_value=200, blur_kernel=5, morph_kernel=3),
    PreprocessingConfig(threshold_method="otsu", threshold_value=200, blur_kernel=5, morph_kernel=3),
    PreprocessingConfig(threshold_method="adaptive", threshold_value=200, blur_kernel=5, morph_kernel=3),
]

fig, axes = plt.subplots(1, len(configs) + 1, figsize=(18, 4))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")

for ax, config in zip(axes[1:], configs):
    result = preprocess_frame(frame, config)
    ax.imshow(result["binary"], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{config.threshold_method}\nused={result['threshold_used']}")
    ax.axis("off")
plt.tight_layout()

## Useful Output Files

- Preprocessing binaries: `outputs/preprocessing/binary/`
- Preprocessing comparisons: `outputs/preprocessing/comparisons/`
- Dataset preview: `outputs/dataset-preview/sample_grid.png`